**# En este archivo se hara la limpieza y transformacion de los datos de la microbioma intestinal**

In [1]:
import os
os.listdir('/kaggle/input/datasets/nico14h/metagenomica')

['SraRunTable(2).csv',
 'dataset.csv',
 'diagnosticos_pacientes.tsv',
 'matriz_final.tsv',
 'matriztotal.tsv',
 'taxonomy.tsv']

In [2]:
#pip install scikit-bio

In [3]:
import pandas as pd
import numpy as np


#tenemos que en matriz, cada una de las columnas son los pacientes, y las filas son los hash
matriz= pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/matriz_final.tsv', sep = '\t', skiprows=1)
matriz= matriz.rename(columns={'#OTU ID': 'Feature ID'})
taxonomia = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/taxonomy.tsv', sep='\t')
#en taxonomia, tendremos: fila(hash) | taxonomia | confianza( de que sea esa bacteria)
# tenemos que los hash NO SE REPITEN, son UNICOS, no hay filas repetidas

#comienzo eliminando lo innecesario de la 3era columna ya que solo me importa el genero de la bacteria, no toda
#la informacion adicional
#ATENCION: es recomendable realizar un analisis con especies de igual manera, no solo genero
def extraer_nombre_bacteria(tax_string):
    # Si la celda está vacía o sin asignar
    if pd.isna(tax_string) or tax_string == 'Unassigned':
        return 'Desconocida'

    # Cortamos el string en una lista de niveles
    niveles = tax_string.split(';')

    # 1. EL FILTRO: Eliminamos por completo la especie ('s__') si es que existe
    niveles_sin_especie = [nivel for nivel in niveles if not nivel.strip().startswith('s__')]

    # 2. BÚSQUEDA: Recorremos lo que quedó de atrás para adelante
    for nivel in reversed(niveles_sin_especie):
        nivel_limpio = nivel.strip()

        # Buscamos el separador biológico
        if '__' in nivel_limpio:
            nombre_puro = nivel_limpio.split('__')[-1].strip()

            # Si el nivel no está vacío (ej. no es solo "g__"), lo retornamos
            if nombre_puro:
                return nombre_puro

    # Fallback de seguridad
    return 'Desconocida'


taxonomia['Bacteria_Limpia'] = taxonomia['Taxon'].apply(extraer_nombre_bacteria)

#luego hago un inner join, usando los hash

df_fusionado = pd.merge(taxonomia[['Feature ID', 'Bacteria_Limpia']], matriz, on='Feature ID', how='inner')

#elimino el hash de cada caso
df_fusionado = df_fusionado.drop(columns=['Feature ID'])

# Agrupamos por el nombre de la bacteria y sumamos los valores numéricos de los pacientes
df_fase1 = df_fusionado.groupby('Bacteria_Limpia').sum().reset_index()

# Verificamos el resultado final
print("Dimensiones de la matriz agrupada:", df_fase1.shape)
print(df_fase1.head())
# se crea el archivo matriztotal.tsv, no se crea en kaggle ya que solo es lectura
#df_fase1.to_csv('/kaggle/input/datasets/nico14h/metagenomica/matriztotal.tsv', sep='\t', index=False)

Dimensiones de la matriz agrupada: (403, 122)
  Bacteria_Limpia  SRR10442269_1  SRR10442270_1  SRR10442271_1  SRR10442272_1  \
0           11-24            0.0            0.0            0.0            0.0   
1          4-29-1            0.0            0.0            0.0            0.0   
2              A2            0.0            0.0            0.0            0.0   
3  ADurb.Bin063-1            0.0            0.0            0.0            0.0   
4        AKAU4049            0.0            0.0            0.0            0.0   

   SRR10442273_1  SRR10442274_1  SRR10442275_1  SRR10442276_1  SRR10442277_1  \
0            0.0            0.0            0.0            0.0            0.0   
1            0.0            0.0            0.0            0.0            0.0   
2            0.0            0.0            0.0            0.0            0.0   
3            0.0            0.0            0.0            0.0            0.0   
4            0.0            0.0            0.0            0.0      

In [4]:
import pandas as pd

# 1. Identificamos cuáles son las columnas de los pacientes 
# (Básicamente, todas las columnas excepto 'Bacteria_Limpia')
columnas_pacientes = df_fase1.columns.drop('Bacteria_Limpia')

# 2. Aislamos SOLO la parte de los números para hacer el cálculo
# Esto crea una vista temporal sin la columna de texto
df_solo_numeros = df_fase1[columnas_pacientes]

# 3. Calculamos la prevalencia FILA por FILA (axis=1)
# (df_solo_numeros > 0) genera True si el valor es mayor a 0
# .mean(axis=1) saca el porcentaje de "True" para cada bacteria (fila)
prevalencia = (df_solo_numeros > 0).mean(axis=1)

# 4. Filtramos el DATAFRAME ORIGINAL
# Le decimos a Pandas: "Del df_fase1 original, guardame solo las filas 
# donde la prevalencia calculada sea mayor o igual a 0.10"
umbral = 0.05
df_filtrado = df_fase1[prevalencia >= umbral]

# Verificamos los resultados
print(f"Cantidad de bacterias antes del filtro: {df_fase1.shape[0]}")
print(f"Cantidad de bacterias después del filtro: {df_filtrado.shape[0]}")
print("\nMuestra del DataFrame limpio:")
print(df_filtrado['Bacteria_Limpia'])


Cantidad de bacterias antes del filtro: 403
Cantidad de bacterias después del filtro: 153

Muestra del DataFrame limpio:
5                           Abiotrophia
13                        Acinetobacter
16                          Actinomyces
17                        Adlercreutzia
19                         Agathobacter
                     ...               
395    [Eubacterium]_xylanophilum_group
396     [Ruminococcus]_gauvreauii_group
397         [Ruminococcus]_gnavus_group
398        [Ruminococcus]_torques_group
402                          uncultured
Name: Bacteria_Limpia, Length: 153, dtype: object


In [5]:
generos_a_eliminar = [
    'Ralstonia', 'Burkholderia', 'Cupriavidus', 'Methylobacterium', 
    'Bradyrhizobium', 'Rhizobium', 'Sphingomonas', 'Novosphingobium',
    'Acetobacter', 'Gluconobacter', 'Oenococcus', 'Cutibacterium'
]

# df_filtrado es tu matriz de 133 bacterias
# Vemos cuántas de las bacterias actuales están en la lista de eliminables
en_la_lista = df_filtrado.index[df_filtrado.index.isin(generos_a_eliminar)]

print(f"De las 133 bacterias, solo {len(en_la_lista)} son contaminantes de tu lista.")
print("Bacterias que se van a borrar:", list(en_la_lista))

De las 133 bacterias, solo 0 son contaminantes de tu lista.
Bacterias que se van a borrar: []


In [6]:
import numpy as np
import pandas as pd

# 1. Extraemos solo los nombres de los pacientes (las columnas numéricas)
df_purificado = df_filtrado.copy()
if 'Bacteria_Limpia' in df_purificado.columns:
    df_purificado = df_purificado.set_index('Bacteria_Limpia')

# 2. Aislamos los datos numéricos y los Transponemos (Pacientes = Filas, Bacterias = Columnas)
matriz_cruda_transpuesta = df_purificado.values.T

# ==========================================
# 3. FUNCIÓN rCLR (Robust Centered Log-Ratio)
# ==========================================
def aplicar_rclr(matriz):
    # Paso A: Enmascaramos los ceros convirtiéndolos temporalmente en NaN 
    # para que no participen en el cálculo de la media geométrica.
    matriz_nan = np.where(matriz == 0, np.nan, matriz)
    
    # Paso B: Calculamos la media geométrica por paciente (ignorando los ceros/NaN)
    media_geom = np.exp(np.nanmean(np.log(matriz_nan), axis=1, keepdims=True))
    
    # Paso C: Aplicamos el log-ratio solo a las bacterias existentes
    matriz_rclr = np.log(matriz_nan / media_geom)
    
    # Paso D: Devolvemos los NaN a su estado original (0.0)
    matriz_rclr_final = np.nan_to_num(matriz_rclr, nan=0.0)
    return matriz_rclr_final

# Aplicamos nuestra función a los datos transpuestos
matriz_rclr_transpuesta = aplicar_rclr(matriz_cruda_transpuesta)

# 4. Volvemos a transponer (Bacterias = Filas, Pacientes = Columnas)
matriz_rclr_final = matriz_rclr_transpuesta.T

# 5. Reconstruimos el DataFrame de Pandas final
df_clr = pd.DataFrame(
    data=matriz_rclr_final, 
    index=df_purificado.index, 
    columns=df_purificado.columns
)

print(f"Ceros reales conservados: {(df_clr == 0.0).sum().sum()}")
print("Muestra del DataFrame con transformación rCLR:")
print(df_clr.head())

Ceros reales conservados: 7720
Muestra del DataFrame con transformación rCLR:
                 SRR10442269_1  SRR10442270_1  SRR10442271_1  SRR10442272_1  \
Bacteria_Limpia                                                               
Abiotrophia                0.0       0.000000       0.000000       0.000000   
Acinetobacter              0.0       0.000000       0.000000       0.000000   
Actinomyces                0.0       0.000000      -1.151516       0.000000   
Adlercreutzia              0.0       0.000000      -1.042317      -1.935149   
Agathobacter               0.0       0.948866       1.608980       1.844484   

                 SRR10442273_1  SRR10442274_1  SRR10442275_1  SRR10442276_1  \
Bacteria_Limpia                                                               
Abiotrophia                0.0       0.000000      -2.638044      -2.787553   
Acinetobacter              0.0       0.000000       0.000000       0.000000   
Actinomyces                0.0      -1.438114      -

In [7]:
# Se utilizara la funcion melt, para poder definir la relacion entre los nodos y el "peso" de la arista
# Se crea la tabla pivotante
# Se van a tener 3 columnas : Bacteria | Paciente | AbundanciaTaxonomica
# En esta etapa tmb se eliminan ceros
df_fase1 = df_clr.copy()
df_fase1 = df_fase1.reset_index()
df_fase1=df_fase1.melt(id_vars=['Bacteria_Limpia'], var_name='Paciente', value_name='AbundanciaTaxonomica')
#print(df_fase1)

#elimino los valores de 0.0
df_fase1.drop(df_fase1[df_fase1['AbundanciaTaxonomica'] == 0.0].index, inplace=True)


#df_fase1['AbundanciaTaxonomica']= np.log1p(df_fase1['AbundanciaTaxonomica'])

# hare que cada paciente este con _1
df_fase1['Paciente'] = df_fase1['Paciente'].apply(lambda x: x if x.endswith('_1') else x + '_1')

# hare que cada paciente este con _1
#df_fase1.to_csv('/content/drive/MyDrive/Tesis/desarrollo/columnasGrafo.tsv', sep='\t', index=False)


print(len(df_fase1['Bacteria_Limpia'].unique()))


153


In [8]:
# Se configura los tensores (numeros de cada paciente y bacteria)
# como se usa pytorch, el mismo solo interpreta numeros
# se hace un diccionario de los pacientes y bacterias
# pacientes (1-126), bacterias (127-N)


pacientes = df_fase1.Paciente.unique()
diccionariopacientes= {nombre: indice for indice, nombre in enumerate(pacientes, start=1)}
print(diccionariopacientes)

bacterias = df_fase1.Bacteria_Limpia.unique()
diccionariobacterias= {nombre: indice for indice, nombre in enumerate(bacterias, start=127)}

pacientes= df_fase1.Paciente
bacterias= df_fase1.Bacteria_Limpia


df_fase1['paciente_id'] = pacientes.map(diccionariopacientes)
df_fase1['bacteria_id'] = bacterias.map(diccionariobacterias)

print(df_fase1)

len(df_fase1['Bacteria_Limpia'].unique())

# 0. Mapeamos los IDs reales a IDs posicionales locales (de 0 a N-1)
df_fase1['paciente_id'] = pd.factorize(df_fase1['paciente_id'])[0]
df_fase1['bacteria_id'] = pd.factorize(df_fase1['bacteria_id'])[0]

print(len(df_fase1['Bacteria_Limpia'].unique()))

{'SRR10442269_1': 1, 'SRR10442270_1': 2, 'SRR10442271_1': 3, 'SRR10442272_1': 4, 'SRR10442273_1': 5, 'SRR10442274_1': 6, 'SRR10442275_1': 7, 'SRR10442276_1': 8, 'SRR10442277_1': 9, 'SRR10442278_1': 10, 'SRR10442279_1': 11, 'SRR10442280_1': 12, 'SRR10442281_1': 13, 'SRR10442282_1': 14, 'SRR10442283_1': 15, 'SRR10442284_1': 16, 'SRR10442285_1': 17, 'SRR10442286_1': 18, 'SRR10442287_1': 19, 'SRR10442288_1': 20, 'SRR10442289_1': 21, 'SRR10442290_1': 22, 'SRR10442291_1': 23, 'SRR10442292_1': 24, 'SRR10442293_1': 25, 'SRR10442294_1': 26, 'SRR10442295_1': 27, 'SRR10442296_1': 28, 'SRR10442297_1': 29, 'SRR10442298_1': 30, 'SRR10442299_1': 31, 'SRR10442300_1': 32, 'SRR10442301_1': 33, 'SRR10442302_1': 34, 'SRR10442303_1': 35, 'SRR10442304_1': 36, 'SRR10442305_1': 37, 'SRR10442306_1': 38, 'SRR10442307_1': 39, 'SRR10442308_1': 40, 'SRR10442309_1': 41, 'SRR10442310_1': 42, 'SRR10442311_1': 43, 'SRR10442312_1': 44, 'SRR10442313_1': 45, 'SRR10442314_1': 46, 'SRR10442315_1': 47, 'SRR10442316_1': 48, 

In [9]:
diagnosticos= pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/diagnosticos_pacientes.tsv', sep='\t')
# a TEA se le asignara el 1 y al HC el 0
#basandome en esto, tenemos que:

diccionarioDiagnosticos= {
    'TEA': 1,
    'HC': 0
}
diagnosticos['Diagnosis'] = diagnosticos['Diagnosis'].map(diccionarioDiagnosticos)

#luego añado la columna de diagnosticos a df_fase1 basandome en los pacientes
# es decir se hace un inner join, por cada nombre de paciente


#diagnosticos.rename(columns={'Patient Name': 'Paciente'}, inplace=True)

diagnosticos['Paciente'] = diagnosticos['Paciente'].str.strip()
df_fase1['Paciente'] = df_fase1['Paciente'].str.strip()

df_fase1['Diagnosticos'] = df_fase1['Paciente'].map(diagnosticos.set_index('Paciente')['Diagnosis'])
pacientes_con_nan = df_fase1[df_fase1['Diagnosticos'].isna()]['Paciente'].tolist()

print(df_fase1)
print(f"Total de pacientes con NaN: {len(pacientes_con_nan)}")
print(f"Ejemplo de 5 pacientes con NaN: {pacientes_con_nan}")

pacientesTEA = df_fase1[df_fase1['Diagnosticos'] == 1]
pacientesHC = df_fase1[df_fase1['Diagnosticos'] == 0]

print(len(pacientesTEA['Paciente'].unique()))

print(len(pacientesHC['Paciente'].unique()))

print(len(df_fase1['Bacteria_Limpia'].unique()))

                       Bacteria_Limpia       Paciente  AbundanciaTaxonomica  \
5                          Akkermansia  SRR10442269_1             -2.663544   
10                        Anaerostipes  SRR10442269_1              0.044506   
14                         Bacteroides  SRR10442269_1              3.197242   
16                     Bifidobacterium  SRR10442269_1              2.215463   
18                             Blautia  SRR10442269_1              2.386312   
...                                ...            ...                   ...   
18507   [Eubacterium]_ventriosum_group  SRR10442395_1             -1.322955   
18509  [Ruminococcus]_gauvreauii_group  SRR10442395_1              0.597114   
18510      [Ruminococcus]_gnavus_group  SRR10442395_1              2.849783   
18511     [Ruminococcus]_torques_group  SRR10442395_1              3.462451   
18512                       uncultured  SRR10442395_1             -0.260710   

       paciente_id  bacteria_id  Diagnosticos  
5  

In [10]:
#creo el dataset df_fase1 para poder usarlo en la otra notebook
df_fase1.to_csv('/kaggle/working/dataset.csv', sep='\t', index=False)